# LSS Brain Decoding — Motif4Limbs — sub-07

## What is LSS (Least Squares Separate)?

For each trial `i`, we fit a **dedicated GLM** on the full run BOLD signal with two task regressors:
- `target`  → HRF convolved with **trial i onset only**  → the beta we extract
- `others`  → HRF convolved with **all other trials lumped** → nuisance

This gives one **beta map** (3D brain volume) per trial — a clean, denoised estimate of the brain response to that specific movement.

```
             voxel_1  voxel_2  ...  voxel_N   label
Trial  1  [   β_1      β_2    ...   β_N   ]   pied_droit
Trial  2  [   β_1      β_2    ...   β_N   ]   main_gauche
...
X : (256, n_voxels)  |  y : (256,)
```

## Why LSS and not the raw BOLD (decoding.ipynb)?
| | Raw BOLD | LSS betas |
|---|---|---|
| HRF handled | Manual offset grid | Automatic (GLM) |
| Motion confounds removed | No | Yes |
| Signal-to-noise | Lower | Higher |
| Standard in papers | Rarely | Yes |

## Why LSS and not LSA (colleague's notebook)?
LSA (Least Squares All) = one big GLM with a unique regressor per trial.  
The problem: with 128 regressors per run, they are highly correlated (each HRF overlaps neighbours).  
This inflates correlations between beta maps → inflated decoding accuracy.  
LSS avoids this by estimating each trial against a clean lumped background.

## Pipeline overview
1. Load BOLD + mask fMRIPrep outputs
2. Load trial onsets from behavioral log (actual timestamps, post-TTL)
3. Pre-mask BOLD with NiftiMasker (once, fast)
4. LSS loop: 128 trials × 2 runs = **256 GLMs**
5. Sanity check: mean beta maps per condition
6. Decoding: LORO-CV, classifiers, confusion matrix, weight maps

In [ ]:
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

from nilearn.maskers import NiftiMasker
from nilearn.masking import intersect_masks
from nilearn import plotting
from nilearn.glm.first_level import make_first_level_design_matrix

from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

print('Imports OK')

In [ ]:
# ==============================================================================
# SETTINGS  (edit this cell only)
# ==============================================================================

SUB   = '07'
SPACE = 'MNI152NLin2009cAsym'
TR    = 1.6   # seconds

# Motion confound columns from fMRIPrep
MOTION_COLS = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']

# Paths (same structure as existing notebooks)
FMRIPREP = Path('/neurospin/motif-stroke/7T_protocol/pilots/derivatives/fmriprep')
BEH_ROOT = Path('/neurospin/motif-stroke/7T_protocol/pilots/Behavioral_data')

FUNC_DIR = FMRIPREP / f'sub-{SUB}' / 'func'
LOG_DIR  = BEH_ROOT / f'sub-{SUB}' / '4limbs' / 'log'

# Results output directory
RESULTS_DIR = Path.home() / '7T-fMRI-Motor-Stroke' / f'results_lss_motif4limbs_V01' / f'sub-{SUB}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Subject  : sub-{SUB}')
print(f'Space    : {SPACE}')
print(f'Results  : {RESULTS_DIR}')

## Step 1 — Load trial events

We use the **behavioral log** (actual timestamps from the experiment computer), NOT the sequence file.

### Why the log and not the sequence?
The sequence file contains **ideal computed times** (blank + stim + pause constants).  
The log gives **actual display timestamps** from the experiment computer.  
The drift accumulates ~25ms per trial → by trial 100 you'd be ~2.5s off (>1 TR). Always use the log.

### Philippe's new fields — what they are and why we ignore them for the GLM
For sub-07, Philippe added movement timing info:
```
FEET:   pied_droit t=1.025s  |  right_pedal_onset:1048  ← RT in ms (button press)
HANDS:  main_gauche t=36.5s  |  left_onset:1047         ← movement started 1047ms after cue
                              |  left_offset:2161        ← movement ended 2161ms after onset
```
These are valuable behavioral metrics (reaction time, movement duration).  
**But for the LSS GLM we use `duration=0` (impulse function).**  

Why duration=0?
- For **decoding**, we want to capture the neural pattern at the moment of movement, not the sustained response
- An impulse gives the sharpest HRF regressor → least overlap between neighboring trials
- Using actual movement duration would create asymmetry (feet have no offset logged, hands do)
- This is the standard choice in decoding papers

In [ ]:
CONDITIONS = ['pied_droit', 'pied_gauche', 'main_gauche', 'main_droite']

def load_events_from_log(run_int):
    """
    Extract trial onsets from the behavioral log.

    Uses actual display timestamps (post-TTL), NOT the sequence file.
    Duration is set to 0 (impulse) — standard for decoding GLMs.

    Philippe's _onset/_offset/_pedal_onset fields are behavioral metrics,
    not used here. They are available for separate RT/vigor analyses.
    """
    logs = sorted(LOG_DIR.glob(f'*sub-{int(SUB)}_run-{run_int}*.csv'))
    if not logs:
        raise FileNotFoundError(f'No log file found for sub-{SUB} run-{run_int} in {LOG_DIR}')

    df = pd.read_csv(logs[0])

    # t0 = TTL pulse = scanner trigger = time 0 for the GLM
    t0 = df[df['event'].isin(['TTL', '116'])].iloc[0]['time']

    # Keep only stimulus display events (the 4 conditions)
    dfb = df[df['event'].isin(CONDITIONS)].copy()
    dfb['onset']      = (dfb['time'] - t0) / 1000.0   # ms → seconds
    dfb['duration']   = 0.0    # impulse: sharpest HRF, least inter-trial overlap
    dfb['trial_type'] = dfb['event']
    return dfb[['onset', 'duration', 'trial_type']].reset_index(drop=True)


# Verify both runs
for run in [1, 2]:
    try:
        ev = load_events_from_log(run)
        print(f'Run {run}: {len(ev)} trials  (duration=0 impulse)')
        print(f'  Conditions: {ev["trial_type"].value_counts().sort_index().to_dict()}')
        print(f'  First onset: {ev["onset"].iloc[0]:.3f}s  |  Last onset: {ev["onset"].iloc[-1]:.3f}s')
        print()
    except FileNotFoundError as e:
        print(f'WARNING: {e}')

## Step 2 — Load BOLD files and build brain mask

In [ ]:
# Discover available runs
runs = []
for run, dir_ in [(1, 'AP'), (2, 'PA')]:
    bold_path = FUNC_DIR / f'sub-{SUB}_task-motif4limbs_dir-{dir_}_run-{run}_space-{SPACE}_desc-preproc_bold.nii.gz'
    mask_path = FUNC_DIR / f'sub-{SUB}_task-motif4limbs_dir-{dir_}_run-{run}_space-{SPACE}_desc-brain_mask.nii.gz'
    conf_path = FUNC_DIR / f'sub-{SUB}_task-motif4limbs_dir-{dir_}_run-{run}_desc-confounds_timeseries.tsv'

    if not bold_path.exists():
        print(f'WARNING: run {run} ({dir_}) BOLD not found — skipping')
        continue

    img = nib.load(str(bold_path))
    print(f'Run {run} ({dir_}): shape={img.shape}  TR={img.header.get_zooms()[-1]:.2f}s')
    runs.append({'run': run, 'dir': dir_, 'bold': bold_path, 'mask': mask_path, 'conf': conf_path})

if not runs:
    raise RuntimeError('No BOLD files found. Check paths.')

# Intersection mask: voxels that are brain in ALL runs
# This ensures every beta map covers the same voxels
mask_img = intersect_masks([str(r['mask']) for r in runs], threshold=1)
n_voxels = int(mask_img.get_fdata().sum())
print(f'\nIntersection mask: {n_voxels:,} voxels')

## Step 3 — Pre-mask BOLD data

We apply the brain mask **once** per run using `NiftiMasker`.  
This gives a 2D matrix `(n_TRs, n_voxels)` per run that we reuse for all 128 LSS GLMs of that run.  
This avoids loading and masking the 4D NIfTI file 128 times — critical for speed.

In [ ]:
masker = NiftiMasker(mask_img=mask_img, standardize=False, verbose=0)
masker.fit(nib.load(str(runs[0]['bold'])))

masked_bold = {}  # run_id → (n_TRs, n_voxels) numpy array

for r in runs:
    print(f'Masking run {r["run"]}...', end=' ', flush=True)
    t0 = time.time()
    img = nib.load(str(r['bold']))
    masked_bold[r['run']] = masker.transform(img)   # (n_TRs, n_voxels)
    print(f'shape {masked_bold[r["run"]].shape}  ({time.time()-t0:.1f}s)')

print('\nBOLD data pre-masked and ready for LSS loop.')

## Step 4 — LSS loop

For each trial `i` in run `r`:
1. Build a design matrix with:
   - `target` regressor: HRF convolved with onset of trial `i` alone
   - `others` regressor: HRF convolved with all **other** trials in that run (same trial_type = `others`)
   - 6 motion regressors + cosine drift (from fMRIPrep confounds)
2. Fit OLS: `β = (X'X)⁻¹ X'Y` — fast numpy computation on pre-masked data
3. Keep only the `target` beta row → shape `(n_voxels,)`

Total: 128 × 2 = **256 GLMs**.

In [ ]:
def build_lss_design(n_trs, events_df, trial_idx, confounds_df):
    """
    Build the LSS design matrix for trial `trial_idx`.

    Parameters
    ----------
    n_trs        : number of TRs in the run
    events_df    : DataFrame with onset, duration, trial_type (all trials in the run)
    trial_idx    : index (0-based) of the target trial
    confounds_df : DataFrame with motion parameters (n_trs × 6)

    Returns
    -------
    dm : pandas DataFrame, the full design matrix
    """
    # Target trial: its own regressor
    target = events_df.iloc[[trial_idx]].copy()
    target['trial_type'] = 'target'

    # All other trials: lumped into one regressor
    # Keeping individual onsets (not one long duration) gives a better HRF estimate
    others = events_df.drop(index=trial_idx).copy()
    others['trial_type'] = 'others'

    events_lss = pd.concat([target, others], ignore_index=True)

    frame_times = np.arange(n_trs) * TR

    dm = make_first_level_design_matrix(
        frame_times   = frame_times,
        events        = events_lss,
        hrf_model     = 'glover',
        drift_model   = 'cosine',
        high_pass     = 0.01,
        add_regs      = confounds_df.values,
        add_reg_names = list(confounds_df.columns),
    )
    return dm


def lss_fit_trial(Y, dm):
    """
    Fit OLS and return the beta for the 'target' regressor.

    Parameters
    ----------
    Y  : (n_trs, n_voxels) masked BOLD data
    dm : design matrix DataFrame

    Returns
    -------
    beta : (n_voxels,) beta estimates for the target regressor
    """
    X = dm.values                                      # (n_trs, n_cols)
    # OLS in one shot across all voxels simultaneously
    # beta shape: (n_cols, n_voxels)
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    target_idx = list(dm.columns).index('target')
    return beta[target_idx]                            # (n_voxels,)


print('LSS helpers defined.')

In [ ]:
beta_maps = []   # will be stacked to (n_trials, n_voxels)
labels    = []   # condition per trial
run_ids   = []   # run number per trial (for LORO CV)

t_total = time.time()

for r in runs:
    print(f"\n{'='*55}")
    print(f"Run {r['run']} ({r['dir']})")
    print(f"{'='*55}")

    events  = load_events_from_log(r['run'])
    conf    = pd.read_csv(r['conf'], sep='\t')[MOTION_COLS].fillna(0)
    Y       = masked_bold[r['run']]    # (n_TRs, n_voxels) — already in memory
    n_trs   = Y.shape[0]
    n_trials = len(events)

    t_run = time.time()

    for i in range(n_trials):
        dm   = build_lss_design(n_trs, events, i, conf)
        beta = lss_fit_trial(Y, dm)

        beta_maps.append(beta)
        labels.append(events.iloc[i]['trial_type'])
        run_ids.append(r['run'])

        if (i + 1) % 32 == 0 or (i + 1) == n_trials:
            elapsed = time.time() - t_run
            eta     = elapsed / (i + 1) * (n_trials - i - 1)
            print(f'  [{i+1:3d}/{n_trials}]  {elapsed:.0f}s elapsed  ETA {eta:.0f}s')

    print(f'  Run {r["run"]} done in {time.time()-t_run:.0f}s')

# Stack into arrays
beta_maps = np.array(beta_maps)   # (n_trials, n_voxels)
labels    = np.array(labels)
run_ids   = np.array(run_ids)

print(f'\nTotal time : {time.time()-t_total:.0f}s')
print(f'beta_maps  : {beta_maps.shape}  (trials × voxels)')
print(f'labels     : {np.unique(labels, return_counts=True)}')
print(f'run_ids    : {dict(zip(*np.unique(run_ids, return_counts=True)))}')

## Step 5 — Sanity check: mean beta map per condition

Average all beta maps of the same condition.  
**Expected**: clear activation in contralateral M1 for each limb (somatotopy):
- Right hand / foot → left M1
- Left hand / foot → right M1
- Hand → lateral M1, foot → medial M1/SMA

If this plot looks flat or noisy, something went wrong upstream.

In [ ]:
label_fr = {
    'main_droite': 'Right hand', 'main_gauche': 'Left hand',
    'pied_droit':  'Right foot', 'pied_gauche': 'Left foot'
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, cond in zip(axes.flat, ['main_droite', 'main_gauche', 'pied_droit', 'pied_gauche']):
    mean_beta = beta_maps[labels == cond].mean(axis=0)   # (n_voxels,)
    beta_img  = masker.inverse_transform(mean_beta)      # back to 3D NIfTI

    plotting.plot_stat_map(
        beta_img, axes=ax,
        title=f'{label_fr[cond]}  (n={np.sum(labels==cond)})',
        display_mode='z', cut_coords=5,
        colorbar=True, threshold=0.05,
    )

plt.suptitle(f'Mean LSS beta map per limb — sub-{SUB} | motif4limbs', fontsize=13)
plt.tight_layout()
out = RESULTS_DIR / 'lss_mean_betas.png'
plt.savefig(str(out), dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Step 6 — Quick GLM-level check: hand vs foot (mean beta contrast)

Before decoding, verify the mean betas show the expected contrast.  
This replicates what your standard GLM would show, but derived from LSS betas.

In [ ]:
hand_mean = beta_maps[np.isin(labels, ['main_droite', 'main_gauche'])].mean(axis=0)
foot_mean = beta_maps[np.isin(labels, ['pied_droit',  'pied_gauche'])].mean(axis=0)

contrast_img = masker.inverse_transform(hand_mean - foot_mean)

plotting.plot_stat_map(
    contrast_img,
    title='Hand > Foot (mean LSS beta contrast)',
    display_mode='z', cut_coords=7, colorbar=True, threshold=0.05,
)
plt.savefig(str(RESULTS_DIR / 'lss_hand_vs_foot_contrast.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Expected: lateral M1 positive (red), medial M1/SMA negative (blue)')

## Step 7 — Decoding

### Cross-validation: Leave-One-Run-Out (LORO)
- Train on run 1 → test on run 2, then flip
- Correct choice: trials within a run are temporally correlated (BOLD autocorrelation)
- Only 2 folds → accuracy estimate is noisy, interpret with caution

### Feature preparation
Z-score each voxel across trials (zero mean, unit variance).  
This removes voxel-level baseline differences and puts all voxels on the same scale.

### Classification problems
| Problem | Chance | Expected |
|---|---|---|
| 4-class (all limbs) | 25% | 50–80% |
| Hand vs Foot | 50% | 80–99% — somatotopy very clear |
| Left vs Right | 50% | 55–75% — requires lateralisation |

In [ ]:
# --- Feature matrix ---
# Z-score per voxel (zero mean, unit variance across trials)
# This is standard preprocessing before linear decoding
X = StandardScaler().fit_transform(beta_maps)   # (n_trials, n_voxels)

# --- Classification problems ---
problems = {
    '4-class':      labels,
    'hand_vs_foot': np.where(np.isin(labels, ['main_droite', 'main_gauche']), 'hand', 'foot'),
    'left_vs_right': np.where(np.isin(labels, ['main_gauche', 'pied_gauche']), 'left', 'right'),
}
chance = {'4-class': 0.25, 'hand_vs_foot': 0.50, 'left_vs_right': 0.50}

# --- Classifiers ---
# Linear only: interpretable weights, appropriate for n_samples << n_features
classifiers = {
    'LinSVC C=0.001': LinearSVC(C=0.001, max_iter=5000),
    'LinSVC C=0.01':  LinearSVC(C=0.01,  max_iter=5000),
    'LinSVC C=0.1':   LinearSVC(C=0.1,   max_iter=5000),
    'LinSVC C=1':     LinearSVC(C=1.0,   max_iter=5000),
    'Ridge a=1':      RidgeClassifier(alpha=1.0),
    'Ridge a=10':     RidgeClassifier(alpha=10.0),
    'Ridge a=100':    RidgeClassifier(alpha=100.0),
    'Ridge a=1000':   RidgeClassifier(alpha=1000.0),
}

logo    = LeaveOneGroupOut()
results = {}   # (problem, classifier) → accuracy
predictions = {}  # (problem, classifier) → y_pred array (for confusion matrix)

print(f'Running {len(problems) * len(classifiers)} CV evaluations...\n')
t0 = time.time()

for prob_name, y_prob in problems.items():
    for clf_name, clf in classifiers.items():
        y_pred = cross_val_predict(clone(clf), X, y_prob, groups=run_ids, cv=logo)
        acc    = accuracy_score(y_prob, y_pred)
        results[(prob_name, clf_name)]     = acc
        predictions[(prob_name, clf_name)] = y_pred
        print(f'{prob_name:14s} | {clf_name:18s} | acc={acc:.1%}')

print(f'\nDone in {time.time()-t0:.0f}s')

In [ ]:
# --- Summary: best classifier per problem ---
print('=' * 60)
print(f'{"Problem":<16} {"Best classifier":<20} {"Accuracy":<10} {"Chance"}')
print('-' * 60)

best_per_problem = {}
for prob_name in problems:
    sub = {k: v for k, v in results.items() if k[0] == prob_name}
    best_key, best_acc = max(sub.items(), key=lambda x: x[1])
    best_per_problem[prob_name] = best_key[1]
    print(f'{prob_name:<16} {best_key[1]:<20} {best_acc:<10.1%} {chance[prob_name]:.0%}')

print('=' * 60)

In [ ]:
# --- Heatmap: classifiers × problems ---
clf_names  = list(classifiers.keys())
prob_names = list(problems.keys())

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, prob_name in zip(axes, prob_names):
    mat = np.array([[results[(prob_name, c)] for c in clf_names]])
    mat = np.array([results[(prob_name, c)] for c in clf_names]).reshape(len(clf_names), 1)

    best_i = mat.argmax()
    im = ax.imshow(mat, vmin=chance[prob_name], vmax=1.0, cmap='RdYlGn', aspect='auto')

    ax.set_yticks(range(len(clf_names)))
    ax.set_yticklabels(clf_names, fontsize=9)
    ax.set_xticks([])
    ax.set_title(f'{prob_name}\nchance={chance[prob_name]:.0%}  best={mat.max():.1%}', fontsize=11)

    for i, acc in enumerate(mat.flatten()):
        color  = 'navy' if i == best_i else 'black'
        weight = 'bold' if i == best_i else 'normal'
        if i == best_i:
            ax.add_patch(plt.Rectangle((-0.5, i - 0.5), 1, 1, fill=False, edgecolor='navy', lw=3))
        ax.text(0, i, f'{acc:.1%}', ha='center', va='center', fontsize=9,
                color=color, fontweight=weight)

    plt.colorbar(im, ax=ax, fraction=0.05)

plt.suptitle(f'Decoding accuracy — sub-{SUB} | LSS betas | LORO-CV', fontsize=13)
plt.tight_layout()
out = RESULTS_DIR / 'lss_decoding_heatmap.png'
plt.savefig(str(out), dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Step 8 — Confusion matrix (4-class, best classifier)

In [ ]:
best_clf_4class = best_per_problem['4-class']
y_pred_best     = predictions[('4-class', best_clf_4class)]
best_acc_4class = results[('4-class', best_clf_4class)]

cond_order = ['main_droite', 'main_gauche', 'pied_droit', 'pied_gauche']
cm = confusion_matrix(labels, y_pred_best, labels=cond_order)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[label_fr[c] for c in cond_order]
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion matrix — 4-class\n{best_clf_4class}  |  acc={best_acc_4class:.1%}')
plt.tight_layout()
out = RESULTS_DIR / 'lss_confusion_4class.png'
plt.savefig(str(out), dpi=120, bbox_inches='tight')
plt.show()

# Per-condition accuracy
print('\nPer-condition accuracy:')
for cond in cond_order:
    idx = labels == cond
    acc = accuracy_score(labels[idx], y_pred_best[idx])
    print(f'  {label_fr[cond]:12s}: {acc:.1%}  ({idx.sum()} trials)')

## Step 9 — Decoding weight maps

Fit the best classifier on **all data** (no CV) and project the weights back to brain space.  

- **Red voxels**: positively weighted for this class (vote FOR this limb)
- **Blue voxels**: negatively weighted (vote AGAINST this limb)

Expected: contralateral M1 weights for each limb.

In [ ]:
clf_final = clone(classifiers[best_clf_4class])
clf_final.fit(X, labels)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (i, cond) in zip(axes.flat, enumerate(clf_final.classes_)):
    w     = clf_final.coef_[i]                     # (n_voxels,)
    w_img = masker.inverse_transform(w)             # back to 3D NIfTI
    thr   = np.percentile(np.abs(w), 90)            # show top 10% most informative voxels

    plotting.plot_stat_map(
        w_img, axes=ax,
        title=f'Weights — {label_fr.get(cond, cond)}',
        display_mode='z', cut_coords=5,
        colorbar=True, threshold=thr,
    )

plt.suptitle(
    f'Per-class decoding weights — {best_clf_4class}\n'
    f'Red=votes FOR this limb  |  Blue=votes AGAINST',
    fontsize=12
)
plt.tight_layout()
out = RESULTS_DIR / 'lss_decoding_weights.png'
plt.savefig(str(out), dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# --- Overall discriminability map: mean |weight| across all 4 classes ---
# = Which voxels are most informative regardless of which limb?
overall_w   = np.abs(clf_final.coef_).mean(axis=0)
overall_img = masker.inverse_transform(overall_w)

plotting.plot_stat_map(
    overall_img,
    title=(
        f'Overall discriminability — top 10% voxels\n'
        f'{best_clf_4class}  |  acc={best_acc_4class:.1%}  |  chance=25%'
    ),
    threshold=np.percentile(overall_w, 90),
    display_mode='z', cut_coords=8, colorbar=True,
)
plt.savefig(str(RESULTS_DIR / 'lss_overall_discriminability.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Expected: bright cluster in bilateral M1 (lateral=hand, medial=foot) and possibly SMA')

---
## Notes and next steps

### Interpreting the results

| Problem | Chance | LSS expected range | Notes |
|---|---|---|---|
| 4-class | 25% | 55–85% | 4 limbs have distinct M1 somatotopy at 7T |
| hand vs foot | 50% | 85–99% | Somatotopy is very clear |
| left vs right | 50% | 55–75% | Contralateral lateralisation |

### Known limitations of this notebook
1. **Only 2 runs** → LORO gives 2 test folds → accuracy estimate is noisy
2. **Whole-brain mask** → ROI (motor cortex atlas) would give cleaner results and faster computation
3. **No permutation test** → the reported accuracy has no p-value — needed before claiming significance
4. **No noise ceiling** → unclear if 70% is good or limited by noise in the data

### Possible next steps
- Add permutation testing (shuffle labels 1000×, get null distribution)
- Add an M1 ROI mask from the Harvard-Oxford atlas
- Extend to more subjects for group-level decoding
- Compare with the raw-BOLD approach in `decoding.ipynb`